# Lab 03 — Edge Detection Techniques and Their Impact on Classification Performance

## README / How to run

This notebook implements the complete Lab 03 workflow using the **same HAM10000 dataset/classes and fixed split** intended for Labs 01–03.

### Included requirements
- Sobel `Gx`, Sobel `Gy`, Sobel magnitude, Prewitt, Laplacian, LoG, and Canny
- Edge-method comparison using images from at least three classes
- Gaussian and salt-and-pepper noise
- Edge detection on original, noisy, Gaussian-smoothed, and median-smoothed inputs
- Table 1: effect of noise and preprocessing
- Canny threshold/kernel analysis
- Table 2: Canny parameter analysis
- Raw vs best-filtered vs best-edge dataset construction
- SVM, Random Forest, KNN, CNN Model 1, CNN Model 2
- Accuracy, precision, recall, F1, training time, inference time
- Table 3: cross-lab classification comparison
- Confusion matrices for raw/filtered/edge inputs
- Metric comparison bar charts
- Discussion and viva answer guide

### Lab dependencies
- Set `SELECTED_CLASSES` to the same classes used in Labs 01–02.
- Set `BEST_LAB02_FILTER` to the best preprocessing method from Lab 02.
- Set `CNN_MODEL_NAMES` to the selected CNNs used for the cross-lab comparison.
- The notebook can determine a useful Canny configuration from its parameter analysis, or you can override it.

### Dataset
Point `DATA_ROOT` to HAM10000 and ensure `HAM10000_metadata.csv` plus the image folders are present.

### Recommended runtime
Google Colab with GPU for the CNN experiments. Classical models use pretrained CNN embeddings to keep SVM/RF/KNN tractable.

In [ ]:
# Optional Colab installation cell.
# !pip -q install tensorflow opencv-python-headless scikit-learn pandas matplotlib

In [ ]:
from pathlib import Path
import time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, DenseNet121

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU(s):", tf.config.list_physical_devices("GPU"))

## 1. Configuration

In [ ]:
DATA_ROOT = Path("/content/HAM10000")
METADATA_CSV = DATA_ROOT / "HAM10000_metadata.csv"

SELECTED_CLASSES = None        # MUST match Labs 01–02; e.g. ["nv", "mel", "bkl"]
MAX_IMAGES_PER_CLASS = None    # e.g. 500 for testing; None for selected full data

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 8
LEARNING_RATE = 1e-4

# Replace with the best method identified by YOUR Lab 02 experiment.
BEST_LAB02_FILTER = "Gaussian"

# Two CNNs for Table 3. Change if your selected models differ.
CNN_MODEL_NAMES = ["EfficientNetB0", "MobileNetV2"]

# Candidate Canny configurations from the task sheet + an extra kernel-size comparison.
CANNY_CONFIGS = [
    {"name": "Canny-1", "low": 30,  "high": 100, "ksize": 3},
    {"name": "Canny-2", "low": 50,  "high": 150, "ksize": 3},
    {"name": "Canny-3", "low": 100, "high": 200, "ksize": 3},
    {"name": "Canny-4", "low": 50,  "high": 150, "ksize": 5},
]

## 2. Dataset preparation — same split for all representations

In [ ]:
def discover_images(root: Path):
    m = {}
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for p in root.rglob(ext):
            m[p.stem] = str(p)
    return m

if not METADATA_CSV.exists():
    raise FileNotFoundError(
        f"Metadata not found: {METADATA_CSV}\n"
        "Upload/mount HAM10000 and update DATA_ROOT."
    )

meta = pd.read_csv(METADATA_CSV)
image_map = discover_images(DATA_ROOT)
meta["path"] = meta["image_id"].map(image_map)
meta = meta.dropna(subset=["path"]).copy()

if SELECTED_CLASSES is not None:
    meta = meta[meta["dx"].isin(SELECTED_CLASSES)].copy()

if MAX_IMAGES_PER_CLASS is not None:
    meta = (
        meta.groupby("dx", group_keys=False)
            .apply(lambda g: g.sample(min(len(g), MAX_IMAGES_PER_CLASS), random_state=SEED))
            .reset_index(drop=True)
    )

class_names = sorted(meta["dx"].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(class_names)}
meta["label"] = meta["dx"].map(class_to_idx).astype(int)

train_df, temp_df = train_test_split(
    meta, test_size=0.30, stratify=meta["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
)

print("Classes:", class_names)
print(f"Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
display(meta["dx"].value_counts().sort_index().to_frame("count"))

In [ ]:
def load_rgb(path, size=IMG_SIZE):
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise ValueError(f"Could not read image: {path}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return cv2.resize(rgb, size, interpolation=cv2.INTER_AREA)

def to_gray(img_rgb):
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

def normalize_u8(arr):
    return cv2.normalize(arr, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

## 3. Task 1 — Comparative edge detection

In [ ]:
def sobel_components(gray):
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)
    return normalize_u8(np.abs(gx)), normalize_u8(np.abs(gy)), normalize_u8(mag)

def prewitt(gray):
    kx = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32)
    ky = np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=np.float32)
    gx = cv2.filter2D(gray, cv2.CV_32F, kx)
    gy = cv2.filter2D(gray, cv2.CV_32F, ky)
    return normalize_u8(cv2.magnitude(gx, gy))

def laplacian(gray):
    return normalize_u8(np.abs(cv2.Laplacian(gray, cv2.CV_32F, ksize=3)))

def log_edges(gray, ksize=5):
    blur = cv2.GaussianBlur(gray, (ksize, ksize), 0)
    return normalize_u8(np.abs(cv2.Laplacian(blur, cv2.CV_32F, ksize=3)))

def canny(gray, low=50, high=150, ksize=3):
    blur = cv2.GaussianBlur(gray, (ksize, ksize), 0)
    return cv2.Canny(blur, low, high)

def edge_maps(img_rgb, canny_low=50, canny_high=150, canny_ksize=3):
    gray = to_gray(img_rgb)
    sx, sy, smag = sobel_components(gray)
    return {
        "Sobel Gx": sx,
        "Sobel Gy": sy,
        "Sobel Magnitude": smag,
        "Prewitt": prewitt(gray),
        "Laplacian": laplacian(gray),
        "LoG": log_edges(gray),
        "Canny": canny(gray, canny_low, canny_high, canny_ksize),
    }

In [ ]:
# Required figure: images from at least three different classes.
chosen_classes = class_names[:3]
rep_rows = [
    meta[meta["dx"] == cls].sample(1, random_state=SEED).iloc[0]
    for cls in chosen_classes
]

for row in rep_rows:
    img = load_rgb(row["path"])
    maps = edge_maps(img)

    # Required compact sequence: Original → Sobel → Prewitt → Laplacian → LoG → Canny
    display_items = [
        ("Original", img),
        ("Sobel", maps["Sobel Magnitude"]),
        ("Prewitt", maps["Prewitt"]),
        ("Laplacian", maps["Laplacian"]),
        ("LoG", maps["LoG"]),
        ("Canny", maps["Canny"]),
    ]

    fig, axes = plt.subplots(1, 6, figsize=(18, 3))
    for ax, (title, im) in zip(axes, display_items):
        ax.imshow(im, cmap=None if im.ndim == 3 else "gray")
        ax.set_title(title)
        ax.axis("off")
    plt.suptitle(f"Class: {row['dx']}")
    plt.tight_layout()
    plt.show()

    # Additional Sobel Gx/Gy required by the lab.
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    for ax, key in zip(axes, ["Sobel Gx", "Sobel Gy", "Sobel Magnitude"]):
        ax.imshow(maps[key], cmap="gray")
        ax.set_title(key); ax.axis("off")
    plt.tight_layout(); plt.show()

## 4. Task 2 — Effect of noise on edge detection

In [ ]:
def add_gaussian_noise(img_rgb, sigma=20):
    noise = np.random.normal(0, sigma, img_rgb.shape).astype(np.float32)
    noisy = np.clip(img_rgb.astype(np.float32) + noise, 0, 255)
    return noisy.astype(np.uint8)

def add_salt_pepper_noise(img_rgb, amount=0.02, salt_vs_pepper=0.5):
    out = img_rgb.copy()
    h, w = out.shape[:2]
    n = int(amount * h * w)

    n_salt = int(n * salt_vs_pepper)
    ys = np.random.randint(0, h, n_salt)
    xs = np.random.randint(0, w, n_salt)
    out[ys, xs] = 255

    n_pepper = n - n_salt
    ys = np.random.randint(0, h, n_pepper)
    xs = np.random.randint(0, w, n_pepper)
    out[ys, xs] = 0
    return out

def gaussian_filter_rgb(img, ksize=5):
    return cv2.GaussianBlur(img, (ksize, ksize), 0)

def median_filter_rgb(img, ksize=5):
    return cv2.medianBlur(img, ksize)

def compute_edge_quality(edge):
    # Objective proxy metrics for comparison.
    edge_pixels = np.count_nonzero(edge)
    density = edge_pixels / edge.size
    # Number of connected edge components; many tiny components can indicate false/noisy edges.
    num_labels, _ = cv2.connectedComponents((edge > 0).astype(np.uint8))
    components = max(num_labels - 1, 0)
    return edge_pixels, density, components

In [ ]:
# Visual comparison on one representative image.
row = rep_rows[0]
original = load_rgb(row["path"])
gauss_noisy = add_gaussian_noise(original)
sp_noisy = add_salt_pepper_noise(original)
gauss_smoothed = gaussian_filter_rgb(gauss_noisy)
median_smoothed = median_filter_rgb(sp_noisy)

variants = {
    "Original": original,
    "Gaussian Noisy": gauss_noisy,
    "S&P Noisy": sp_noisy,
    "Gaussian Noisy + Gaussian Filter": gauss_smoothed,
    "S&P Noisy + Median Filter": median_smoothed,
}

fig, axes = plt.subplots(1, len(variants), figsize=(18, 4))
for ax, (name, im) in zip(axes, variants.items()):
    ax.imshow(im); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Table 1: Effect of Noise and Preprocessing on Edge Detection
# We compute objective edge-density/component statistics and leave a concise observation column.

table1_rows = []

def selected_edge(gray, detector):
    if detector == "Sobel":
        return sobel_components(gray)[2]
    if detector == "Prewitt":
        return prewitt(gray)
    if detector == "Laplacian":
        return laplacian(gray)
    if detector == "LoG":
        return log_edges(gray)
    if detector == "Canny":
        return canny(gray, 50, 150, 3)
    raise ValueError(detector)

cases = [
    ("Sobel", "Original", "None", "None", original),
    ("Sobel", "Noisy", "Gaussian", "None", gauss_noisy),
    ("Sobel", "Noisy", "Salt & Pepper", "None", sp_noisy),
    ("Sobel", "Noisy", "Gaussian", "Gaussian Filter", gauss_smoothed),
    ("Sobel", "Noisy", "Salt & Pepper", "Median Filter", median_smoothed),
    ("Prewitt", "Original", "None", "None", original),
    ("Laplacian", "Original", "None", "None", original),
    ("LoG", "Noisy", "Gaussian", "Gaussian Filter", gauss_smoothed),
    ("Canny", "Original", "None", "Built-in smoothing", original),
    ("Canny", "Noisy", "Gaussian", "Gaussian Filter", gauss_smoothed),
    ("Canny", "Noisy", "Salt & Pepper", "Median Filter", median_smoothed),
]

for detector, input_type, noise_type, prep, im in cases:
    edge = selected_edge(to_gray(im), detector)
    n_edges, density, components = compute_edge_quality(edge)
    sensitivity = "Higher" if ("Noisy" in input_type and density > 0.15) else "Lower/Moderate"
    quality = "Cleaner/Sparser" if density < 0.15 else "Dense/Noisy"
    obs = f"edge_density={density:.3f}; components={components}"
    table1_rows.append({
        "Edge Detector": detector,
        "Input Image": input_type,
        "Noise Type": noise_type,
        "Preprocessing": prep,
        "Edge Quality": quality,
        "Noise Sensitivity": sensitivity,
        "Observations": obs,
    })

table1 = pd.DataFrame(table1_rows)
display(table1)

### Observation prompts for Task 2
When interpreting Table 1 and the displayed edge maps, explicitly comment on:
- edge continuity
- edge sharpness
- false edges
- broken edges
- noise sensitivity
- effect of smoothing

## 5. Task 3 — Canny parameter analysis

In [ ]:
def canny_analysis_on_image(img_rgb, configs=CANNY_CONFIGS):
    gray = to_gray(img_rgb)
    rows = []
    fig, axes = plt.subplots(1, len(configs), figsize=(4 * len(configs), 4))

    for ax, cfg in zip(np.atleast_1d(axes), configs):
        edge = canny(gray, cfg["low"], cfg["high"], cfg["ksize"])
        count, density, components = compute_edge_quality(edge)
        quality = "Sparse" if density < 0.05 else ("Moderate" if density < 0.15 else "Dense")
        rows.append({
            "Configuration": cfg["name"],
            "Low Threshold": cfg["low"],
            "High Threshold": cfg["high"],
            "Kernel Size": f'{cfg["ksize"]}×{cfg["ksize"]}',
            "Edge Quality": quality,
            "Number of Detected Edges": count,
            "Observation": f"density={density:.3f}, components={components}",
        })
        ax.imshow(edge, cmap="gray")
        ax.set_title(cfg["name"])
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    return pd.DataFrame(rows)

table2 = canny_analysis_on_image(original)
display(table2)

In [ ]:
# Select a useful Canny configuration.
# Automatic heuristic: prefer a moderate edge density close to 0.10.
def density_for_cfg(img_rgb, cfg):
    edge = canny(to_gray(img_rgb), cfg["low"], cfg["high"], cfg["ksize"])
    return np.count_nonzero(edge) / edge.size

cfg_scores = []
for cfg in CANNY_CONFIGS:
    densities = [density_for_cfg(load_rgb(r["path"]), cfg) for r in rep_rows]
    mean_density = float(np.mean(densities))
    score = abs(mean_density - 0.10)
    cfg_scores.append((score, mean_density, cfg))

cfg_scores.sort(key=lambda x: x[0])
BEST_CANNY = cfg_scores[0][2]
print("Automatically selected Canny configuration:", BEST_CANNY)
print("Mean edge density:", cfg_scores[0][1])

## 6. Task 4 — Raw, filtered, and edge dataset representations

In [ ]:
def apply_lab02_filter(img_rgb, name):
    if name == "No Filter":
        return img_rgb
    if name == "Average":
        return cv2.blur(img_rgb, (5, 5))
    if name == "Gaussian":
        return cv2.GaussianBlur(img_rgb, (5, 5), 0)
    if name == "Median":
        return cv2.medianBlur(img_rgb, 5)
    if name == "Sharpening":
        kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32)
        return cv2.filter2D(img_rgb, -1, kernel)
    if name == "Sobel":
        g = to_gray(img_rgb)
        sx, sy, sm = sobel_components(g)
        return cv2.cvtColor(sm, cv2.COLOR_GRAY2RGB)
    raise ValueError(name)

def represent_image(img_rgb, representation):
    if representation == "Raw":
        return img_rgb
    if representation == "Filtered":
        return apply_lab02_filter(img_rgb, BEST_LAB02_FILTER)
    if representation == "Edge":
        e = canny(
            to_gray(img_rgb),
            BEST_CANNY["low"],
            BEST_CANNY["high"],
            BEST_CANNY["ksize"]
        )
        return cv2.cvtColor(e, cv2.COLOR_GRAY2RGB)
    raise ValueError(representation)

REPRESENTATIONS = ["Raw", "Filtered", "Edge"]

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax, rep in zip(axes, REPRESENTATIONS):
    ax.imshow(represent_image(original, rep))
    ax.set_title(rep); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Feature extraction for SVM, Random Forest, and KNN

In [ ]:
# Use a frozen ImageNet CNN as a common feature extractor for all three input representations.
# This keeps the classical classifiers computationally manageable while preserving the same split.

feature_base = MobileNetV2(
    include_top=False, weights="imagenet",
    input_shape=(*IMG_SIZE, 3), pooling="avg"
)
feature_base.trainable = False

def dataframe_to_array(df, representation):
    X = []
    for p in df["path"]:
        img = load_rgb(p)
        img = represent_image(img, representation).astype(np.float32)
        img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
        X.append(img)
    return np.stack(X), df["label"].to_numpy()

def extract_features(df, representation):
    X_img, y = dataframe_to_array(df, representation)
    feats = feature_base.predict(X_img, batch_size=BATCH_SIZE, verbose=1)
    return feats, y

In [ ]:
# Cache features so each classical classifier sees the exact same data.
feature_cache = {}

for rep in REPRESENTATIONS:
    print("\nExtracting:", rep)
    Xtr, ytr = extract_features(train_df, rep)
    Xva, yva = extract_features(val_df, rep)
    Xte, yte = extract_features(test_df, rep)
    feature_cache[rep] = (Xtr, ytr, Xva, yva, Xte, yte)

## 8. Classical classifier training and evaluation

In [ ]:
def classification_metrics(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": p,
        "Recall": r,
        "F1-score": f1,
    }

CLASSIFIERS = {
    "SVM": SVC(kernel="rbf", C=10, gamma="scale"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=SEED, n_jobs=-1
    ),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

all_results = []
all_predictions = {}

for clf_name, clf in CLASSIFIERS.items():
    for rep in REPRESENTATIONS:
        Xtr, ytr, Xva, yva, Xte, yte = feature_cache[rep]

        t0 = time.perf_counter()
        clf.fit(Xtr, ytr)
        train_time = time.perf_counter() - t0

        t1 = time.perf_counter()
        pred = clf.predict(Xte)
        infer_time = time.perf_counter() - t1

        metrics = classification_metrics(yte, pred)
        all_results.append({
            "Model / Classifier": clf_name,
            "Representation": rep,
            **metrics,
            "Training Time (s)": train_time,
            "Inference Time (ms)": 1000 * infer_time / max(len(yte), 1),
        })
        all_predictions[(clf_name, rep)] = pred

## 9. CNN Model 1 and CNN Model 2

In [ ]:
CNN_FACTORY = {
    "EfficientNetB0": EfficientNetB0,
    "MobileNetV2": MobileNetV2,
    "DenseNet121": DenseNet121,
}

def make_tf_dataset(df, representation, training=False):
    paths = df["path"].astype(str).values
    labels = df["label"].astype(np.int32).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)

    def _py(path_bytes, rep_bytes):
        p = path_bytes.numpy().decode()
        rep = rep_bytes.numpy().decode()
        img = represent_image(load_rgb(p), rep).astype(np.float32)
        return img

    def mapper(path, label):
        img = tf.py_function(_py, [path, tf.constant(representation)], tf.float32)
        img.set_shape((*IMG_SIZE, 3))
        img = tf.keras.applications.imagenet_utils.preprocess_input(img, mode="torch")
        return img, label

    return ds.map(mapper, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def build_cnn(name):
    base = CNN_FACTORY[name](
        include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3)
    )
    base.trainable = False
    inp = layers.Input(shape=(*IMG_SIZE, 3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.25)(x)
    out = layers.Dense(len(class_names), activation="softmax")(x)
    m = models.Model(inp, out)
    m.compile(
        optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return m

In [ ]:
cnn_histories = {}

for cnn_name in CNN_MODEL_NAMES:
    if cnn_name not in CNN_FACTORY:
        raise ValueError(f"Unsupported CNN name: {cnn_name}")

    for rep in REPRESENTATIONS:
        print("\n" + "=" * 80)
        print(cnn_name, "|", rep)

        train_ds = make_tf_dataset(train_df, rep, training=True)
        val_ds = make_tf_dataset(val_df, rep)
        test_ds = make_tf_dataset(test_df, rep)

        tf.keras.backend.clear_session()
        model = build_cnn(cnn_name)

        t0 = time.perf_counter()
        hist = model.fit(
            train_ds, validation_data=val_ds, epochs=EPOCHS,
            callbacks=[callbacks.EarlyStopping(
                monitor="val_loss", patience=2, restore_best_weights=True
            )],
            verbose=1
        )
        train_time = time.perf_counter() - t0

        t1 = time.perf_counter()
        probs = model.predict(test_ds, verbose=0)
        infer_time = time.perf_counter() - t1
        pred = probs.argmax(axis=1)

        metrics = classification_metrics(test_df["label"].to_numpy(), pred)
        all_results.append({
            "Model / Classifier": cnn_name,
            "Representation": rep,
            **metrics,
            "Training Time (s)": train_time,
            "Inference Time (ms)": 1000 * infer_time / max(len(test_df), 1),
        })
        all_predictions[(cnn_name, rep)] = pred
        cnn_histories[(cnn_name, rep)] = hist.history

## 10. Task 5 — Cross-lab classification performance comparison

In [ ]:
long_results = pd.DataFrame(all_results)
display(long_results)

# Required Table 3 layout: raw/filtered/edge accuracies + precision/recall/F1/time.
acc_pivot = long_results.pivot(
    index="Model / Classifier", columns="Representation", values="Accuracy"
).reset_index()

summary_rows = []
for model_name in long_results["Model / Classifier"].unique():
    sub = long_results[long_results["Model / Classifier"] == model_name]
    # Use Edge row for the single precision/recall/F1/time columns in the provided table format.
    edge_row = sub[sub["Representation"] == "Edge"].iloc[0]
    row = {
        "Model / Classifier": model_name,
        "Accuracy Raw (Lab 1)": float(sub[sub["Representation"]=="Raw"]["Accuracy"].iloc[0]),
        "Accuracy Filtered (Lab 2)": float(sub[sub["Representation"]=="Filtered"]["Accuracy"].iloc[0]),
        "Accuracy Edge (Lab 3)": float(edge_row["Accuracy"]),
        "Precision": float(edge_row["Precision"]),
        "Recall": float(edge_row["Recall"]),
        "F1-Score": float(edge_row["F1-score"]),
        "Training Time (s)": float(edge_row["Training Time (s)"]),
        "Inference Time (ms)": float(edge_row["Inference Time (ms)"]),
    }
    summary_rows.append(row)

table3 = pd.DataFrame(summary_rows)
display(table3)

long_results.to_csv("lab03_all_representation_results.csv", index=False)
table1.to_csv("lab03_table1_noise_preprocessing.csv", index=False)
table2.to_csv("lab03_table2_canny_parameters.csv", index=False)
table3.to_csv("lab03_table3_cross_lab_comparison.csv", index=False)

## 11. Task 6 — Visual comparison of classification results

In [ ]:
# Best model is selected by its best observed F1-score across the three representations.
best_model_name = (
    long_results.groupby("Model / Classifier")["F1-score"]
    .max().idxmax()
)
print("Best observed model:", best_model_name)

y_true = test_df["label"].to_numpy()

for rep in REPRESENTATIONS:
    pred = all_predictions[(best_model_name, rep)]
    cm = confusion_matrix(y_true, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(7, 6))
    disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
    ax.set_title(f"{best_model_name} — {rep}")
    plt.tight_layout()
    plt.show()

In [ ]:
best_rows = long_results[long_results["Model / Classifier"] == best_model_name].copy()
metric_plot = best_rows.set_index("Representation")[["Accuracy", "Precision", "Recall", "F1-score"]]
display(metric_plot)

metric_plot.plot(kind="bar", figsize=(10, 5))
plt.title(f"Raw vs Filtered vs Edge — {best_model_name}")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 12. CNN training/validation curves

In [ ]:
for (cnn_name, rep), h in cnn_histories.items():
    plt.figure(figsize=(6, 4))
    plt.plot(h["accuracy"], label="train")
    plt.plot(h["val_accuracy"], label="validation")
    plt.title(f"{cnn_name} — {rep} — Accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
    plt.tight_layout(); plt.show()

    plt.figure(figsize=(6, 4))
    plt.plot(h["loss"], label="train")
    plt.plot(h["val_loss"], label="validation")
    plt.title(f"{cnn_name} — {rep} — Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
    plt.tight_layout(); plt.show()

## 13. Discussion questions — answer guide

Use your generated figures and tables as the primary evidence.

1. **Which edge detector was most sensitive to noise?**  
   Compare false-edge density, fragmentation, and connected components on noisy images. Second-order Laplacian responses are often particularly noise-sensitive because differentiation emphasizes rapid intensity fluctuations, but your conclusion should follow your actual maps.

2. **How did Gaussian and Median filtering affect detected edges?**  
   Gaussian filtering generally suppresses Gaussian noise and smooths fine detail; median filtering is especially effective for impulse/salt-and-pepper noise while preserving many boundaries. Compare the before/after edge maps and Table 1.

3. **How did Canny low/high thresholds change edge number and quality?**  
   Lower thresholds usually retain more weak edges and may include more false edges. Higher thresholds normally create sparser edge maps and may discard weak true boundaries. Use Table 2's detected-edge counts.

4. **Did edge-only images improve or reduce classification accuracy versus raw images?**  
   Compare the `Raw` and `Edge` rows in `long_results`. Do not assume the direction before running.

5. **What information is lost in edge maps?**  
   Color, absolute intensity, much of texture, pigmentation patterns, smooth gradients, and region-level appearance can be discarded or strongly compressed.

6. **Why let CNNs learn edge-like features automatically?**  
   CNN filters are optimized for the dataset/task and can combine edges with color, texture, shape, and higher-level patterns. Fixed hand-crafted edge maps may remove potentially discriminative information before the network sees it.

7. **Which representation was most useful: raw, filtered, or edge?**  
   Compare accuracy/precision/recall/F1 for each model. Support the answer with Table 3 and the best-model bar chart.

## 14. Viva questions — concise answers

1. **What is an edge?** A location where image intensity changes sharply, often corresponding to a boundary or structural transition.  
2. **First-order vs second-order detection?** First-order methods use intensity gradients; second-order methods use the second derivative, such as the Laplacian.  
3. **Sobel Gx vs Gy?** `Gx` responds mainly to vertical boundaries (horizontal intensity change); `Gy` responds mainly to horizontal boundaries.  
4. **Why is Laplacian noise-sensitive?** Second derivatives strongly amplify high-frequency fluctuations, including noise.  
5. **Purpose of Gaussian smoothing?** Reduce noise/high-frequency variation before derivatives or thresholding.  
6. **Main advantage of Canny?** It combines smoothing, gradient estimation, non-maximum suppression, and hysteresis thresholding for thin, connected, noise-robust edges.  
7. **Canny low/high thresholds?** They define hysteresis: strong edges exceed the high threshold; weak edges between thresholds are retained when connected to strong edges.  
8. **Gaussian vs salt-and-pepper noise?** Gaussian noise adds continuous random intensity variation; salt-and-pepper noise replaces pixels with extreme dark/bright values.  
9. **Why median filtering for salt-and-pepper noise?** The median rejects isolated extreme values while preserving edges better than simple averaging.  
10. **Why can edge detection reduce classification performance?** It can discard color, texture, intensity and region information important for the class.  
11. **Can CNNs learn edge features automatically?** Yes. Early CNN layers commonly learn local gradient/edge-like filters when useful for the task.  
12. **Why might raw images outperform edge-only images?** Raw images retain the full visual signal, allowing the CNN to learn which color, texture, boundary and intensity features matter.

## 15. Submission checklist

The Lab 03 deliverables require:
- complete notebook/code
- original, noisy, filtered and edge images
- Sobel, Prewitt, Laplacian, LoG and Canny results
- confusion matrices
- performance comparison graphs
- completed Tables 1–3
- a short report with Introduction, Methodology, Experimental Setup, Results, Discussion, Conclusion and References

This notebook generates the experimental material needed for those deliverables. The README section at the top can be copied into a standalone `README.md` if your repository later requires one.